<div style="background: linear-gradient(135deg, #0a0a1a 0%, #0d1b2a 50%, #1b2838 100%); padding: 40px; border-radius: 15px; margin-bottom: 20px;">
  <h1 style="color: #f0c040; font-size: 2.5em; text-align: center; margin-bottom: 10px;">🪐 Physics-Informed Neural Networks</h1>
  <h2 style="color: #7fb3d3; font-size: 1.6em; text-align: center; margin-bottom: 20px;">Kepler Orbit Reconstruction</h2>
  <p style="color: #ccc; text-align: center; font-size: 1.1em;">Using Newton's law of gravitation as a physics constraint to extrapolate orbital trajectories from sparse data</p>
  <hr style="border-color: #f0c040; margin: 20px 0;">
  <div style="display: flex; justify-content: center; gap: 15px; flex-wrap: wrap;">
    <span style="background: #e74c3c; color: white; padding: 5px 15px; border-radius: 20px; font-size: 0.9em;">PyTorch</span>
    <span style="background: #2980b9; color: white; padding: 5px 15px; border-radius: 20px; font-size: 0.9em;">SciPy ODE</span>
    <span style="background: #8e44ad; color: white; padding: 5px 15px; border-radius: 20px; font-size: 0.9em;">Autograd</span>
    <span style="background: #16a085; color: white; padding: 5px 15px; border-radius: 20px; font-size: 0.9em;">Newton Gravity</span>
    <span style="background: #d35400; color: white; padding: 5px 15px; border-radius: 20px; font-size: 0.9em;">Kepler's Laws</span>
  </div>
</div>

## 📋 Table of Contents

1. [🌌 Introduction & Motivation](#intro)
2. [🔭 Kepler's Laws & Orbital Mechanics](#kepler)
3. [📐 Governing Equations](#equations)
4. [📊 Numerical Solution & Orbital Visualisation](#numerical)
5. [⚡ Orbital Conservation Laws](#conservation)
6. [🧠 PINN Architecture & Loss Function](#pinn-theory)
7. [🏗️ Network Implementation](#network)
8. [📉 Standard NN Baseline](#ann)
9. [🚀 PINN Training](#pinn)
10. [📈 Loss Curves](#loss)
11. [🔍 Final Comparison & Error Analysis](#comparison)
12. [💎 Conservation Law Verification](#conservation-check)
13. [🎬 Training Animation](#animation)
14. [🏁 Conclusions](#conclusions)

<a id='intro'></a>
## 🌌 1. Introduction & Motivation

Predicting planetary orbits is one of the oldest problems in physics — and one of the clearest demonstrations of why **physics knowledge matters** in machine learning.

### The Challenge

Suppose we track a planet's position for only **the first 40% of its orbit** (think: a telescope window). Can we predict the rest?

- A **standard neural network** interpolates the training data but extrapolates poorly — it doesn't know orbits are closed ellipses.
- A **PINN**, constrained by Newton's law of gravity, correctly reconstructs the full ellipse — even where it has never seen data.

<div style="background: #fff3cd; border-left: 5px solid #ffc107; padding: 15px; border-radius: 5px; margin: 15px 0;">
<strong>🔑 Key Physics:</strong> Newton's law of gravitation gives us the exact accelerations at every point:
$$\ddot{\mathbf{r}} = -\frac{GM}{r^3}\mathbf{r}$$
This is a powerful constraint — the PINN is penalised whenever its predicted trajectory violates this law.
</div>

<a id='kepler'></a>
## 🔭 2. Kepler's Laws & Orbital Mechanics

Johannes Kepler (1609–1619) discovered three empirical laws of planetary motion:

| Law | Statement |
|-----|----------|
| **1st** | Planets move in **ellipses** with the Sun at one focus |
| **2nd** | A planet sweeps out **equal areas** in equal times (angular momentum conservation) |
| **3rd** | $T^2 \propto a^3$ — orbital period squared ∝ semi-major axis cubed |

These laws follow from Newton's universal law of gravitation:
$$\mathbf{F} = -\frac{GMm}{r^2}\hat{r}$$

### Orbital Elements

An ellipse with semi-major axis $a$ and eccentricity $e$:
- **Periapsis** (closest point): $r_{\min} = a(1-e)$
- **Apoapsis** (farthest point): $r_{\max} = a(1+e)$
- **Period**: $T = 2\pi\sqrt{a^3 / GM}$

At periapsis, the velocity is purely tangential (vis-viva equation):
$$v_{\text{periapsis}} = \sqrt{\frac{GM(1+e)}{a(1-e)}}$$

<a id='equations'></a>
## 📐 3. Governing Equations

In 2D Cartesian coordinates $(x, y)$ with the central body at the origin:

$$\boxed{\ddot{x} = -\frac{GM \cdot x}{r^3}, \qquad \ddot{y} = -\frac{GM \cdot y}{r^3}}$$

where $r = \sqrt{x^2 + y^2}$.

This is a **system of two coupled, nonlinear, second-order ODEs** — equivalent to a 4D first-order system:

$$\frac{d}{dt}\begin{pmatrix}x \\ y \\ v_x \\ v_y\end{pmatrix} = \begin{pmatrix}v_x \\ v_y \\ -GMx/r^3 \\ -GMy/r^3\end{pmatrix}$$

### Conserved Quantities

$$E = \frac{1}{2}(v_x^2 + v_y^2) - \frac{GM}{r} = \text{const} \qquad \text{(total energy)}$$
$$L = x v_y - y v_x = \text{const} \qquad \text{(angular momentum)}$$

A well-trained PINN should **conserve these quantities** — even though we never explicitly enforce them.

### Parameters (dimensionless units)

| Parameter | Value |
|-----------|-------|
| $GM$ | 1.0 |
| Semi-major axis $a$ | 1.0 |
| Eccentricity $e$ | 0.5 |
| Initial position $(x_0, y_0)$ | $(0.5,\; 0)$ |
| Initial velocity $(v_{x0}, v_{y0})$ | $(0,\; \sqrt{3GM})$ |

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import os
import numpy as np
import torch
import torch.nn as nn
from scipy.integrate import odeint
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.animation import FuncAnimation
from matplotlib import rcParams
from matplotlib.patches import FancyArrowPatch
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

os.makedirs('plots', exist_ok=True)

rcParams['font.family'] = 'DejaVu Sans'
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False

COLORS = {
    'exact':   '#2ecc71',
    'nn':      '#3498db',
    'pinn':    '#e74c3c',
    'data':    '#f39c12',
    'physics': '#9b59b6',
    'energy':  '#1abc9c',
    'angmom':  '#e67e22',
}

print('✅ Imports OK')
print(f'   PyTorch : {torch.__version__}   NumPy : {np.__version__}')

In [ ]:
# ── Physical parameters (dimensionless units) ──────────────────────────────────
GM           = 1.0
ECCENTRICITY = 0.5
a            = 1.0   # semi-major axis

# Initial conditions at periapsis
x0  = a * (1 - ECCENTRICITY)
y0  = 0.0
vx0 = 0.0
vy0 = np.sqrt(GM * (1 + ECCENTRICITY) / (a * (1 - ECCENTRICITY)))

T_orbit         = 2 * np.pi * np.sqrt(a**3 / GM)  # Kepler's 3rd law
T_SIM           = 1.5 * T_orbit
END_TRAIN_FRAC  = 0.4   # fraction of orbit seen during training
N_DENSE         = 1000

print(f'Orbital period        T  = {T_orbit:.4f}')
print(f'Simulation time       T_sim = {T_SIM:.4f}  ({1.5} periods)')
print(f'Eccentricity          e  = {ECCENTRICITY}')
print(f'Periapsis             r_min = {a*(1-ECCENTRICITY):.3f}')
print(f'Apoapsis              r_max = {a*(1+ECCENTRICITY):.3f}')
print(f'Initial velocity      vy0  = {vy0:.4f}')

<a id='numerical'></a>
## 📊 4. Numerical Solution & Orbital Visualisation

In [ ]:
def kepler_ode(state, t):
    x, y, vx, vy = state
    r  = np.sqrt(x**2 + y**2)
    return [vx, vy, -GM*x/r**3, -GM*y/r**3]

t_dense   = np.linspace(0, T_SIM, N_DENSE)
sol       = odeint(kepler_ode, [x0, y0, vx0, vy0], t_dense, rtol=1e-10, atol=1e-12)
x_exact   = sol[:, 0];  y_exact  = sol[:, 1]
vx_exact  = sol[:, 2];  vy_exact = sol[:, 3]
r_exact   = np.sqrt(x_exact**2 + y_exact**2)

print(f'Solution shape: {sol.shape}')
print(f'r_min = {r_exact.min():.4f}  (expected {a*(1-ECCENTRICITY):.4f})')
print(f'r_max = {r_exact.max():.4f}  (expected {a*(1+ECCENTRICITY):.4f})')

In [ ]:
fig = plt.figure(figsize=(15, 6))
gs  = gridspec.GridSpec(1, 3, wspace=0.35)

# ─── Panel A: orbital ellipse ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
# Color-code by speed
speed = np.sqrt(vx_exact**2 + vy_exact**2)
sc = ax1.scatter(x_exact, y_exact, c=speed, cmap='plasma', s=8, zorder=3)
plt.colorbar(sc, ax=ax1, label='Speed')
ax1.scatter([0], [0], s=400, color='gold', marker='*', zorder=6,
            edgecolors='orange', lw=2, label='Central body (focus)')
ax1.scatter([x0], [y0], s=120, color=COLORS['data'], marker='D', zorder=5,
            edgecolors='k', lw=0.8, label='Periapsis (start)')
ax1.annotate('Periapsis\n(fast)', xy=(x0, y0), xytext=(x0+0.05, 0.25),
             fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))
ax1.annotate('Apoapsis\n(slow)', xy=(-a*(1+ECCENTRICITY), 0), xytext=(-1.4, -0.35),
             fontsize=9, arrowprops=dict(arrowstyle='->', color='gray'))
ax1.set_aspect('equal')
ax1.set_xlabel('x', fontsize=12);  ax1.set_ylabel('y', fontsize=12)
ax1.set_title('(A) Kepler Ellipse\n(color = speed)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9, framealpha=0.9)

# ─── Panel B: x(t), y(t) ──────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.plot(t_dense, x_exact, color=COLORS['exact'],  lw=2,   label='x(t)')
ax2.plot(t_dense, y_exact, color=COLORS['energy'], lw=2,   label='y(t)')
ax2.plot(t_dense, r_exact, color=COLORS['pinn'],   lw=1.5, ls=':', label='r(t)')
ax2.axvspan(0, END_TRAIN_FRAC*T_SIM, alpha=0.1, color='orange', label='Training window')
ax2.set_xlabel('Time', fontsize=12);  ax2.set_ylabel('Coordinate', fontsize=12)
ax2.set_title('(B) Coordinates & Radius vs Time', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10, framealpha=0.9); ax2.grid(alpha=0.3)

# ─── Panel C: velocity vectors ────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
stride = 50
ax3.plot(x_exact, y_exact, color=COLORS['exact'], lw=1.5, alpha=0.4)
ax3.quiver(x_exact[::stride], y_exact[::stride],
           vx_exact[::stride], vy_exact[::stride],
           speed[::stride], cmap='cool', scale=15, width=0.005)
ax3.scatter([0], [0], s=300, color='gold', marker='*', zorder=5, edgecolors='orange')
ax3.set_aspect('equal')
ax3.set_xlabel('x', fontsize=12);  ax3.set_ylabel('y', fontsize=12)
ax3.set_title('(C) Velocity Field Along Orbit', fontsize=12, fontweight='bold')

plt.suptitle(fr'Kepler Orbit — Eccentricity $e={ECCENTRICITY}$, $GM={GM}$',
             fontsize=14, fontweight='bold')
plt.savefig('plots/01_kepler_orbit.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='conservation'></a>
## ⚡ 5. Orbital Conservation Laws

In [ ]:
E_exact = 0.5 * (vx_exact**2 + vy_exact**2) - GM / r_exact
L_exact = x_exact * vy_exact - y_exact * vx_exact   # z-component of L

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(t_dense, E_exact, color=COLORS['energy'], lw=2)
ax.axhline(E_exact[0], color='gray', ls='--', lw=1, alpha=0.7)
ax.fill_between(t_dense, E_exact, E_exact[0], alpha=0.15, color=COLORS['energy'])
ax.set_xlabel('Time', fontsize=12); ax.set_ylabel('Total Energy $E$', fontsize=12)
ax.set_title(fr'Total Energy  (should be constant = {E_exact[0]:.5f})', fontweight='bold')
ax.set_ylim(E_exact[0] - 0.01, E_exact[0] + 0.01)
ax.grid(alpha=0.3)
ax.text(0.5, 0.05, f'Max drift: {np.max(np.abs(E_exact - E_exact[0])):.2e}',
        transform=ax.transAxes, ha='center', fontsize=10, color='gray')

ax2 = axes[1]
ax2.plot(t_dense, L_exact, color=COLORS['angmom'], lw=2)
ax2.axhline(L_exact[0], color='gray', ls='--', lw=1, alpha=0.7)
ax2.fill_between(t_dense, L_exact, L_exact[0], alpha=0.15, color=COLORS['angmom'])
ax2.set_xlabel('Time', fontsize=12); ax2.set_ylabel('Angular Momentum $L$', fontsize=12)
ax2.set_title(fr'Angular Momentum (Kepler 2nd law, = {L_exact[0]:.5f})', fontweight='bold')
ax2.set_ylim(L_exact[0] - 0.01, L_exact[0] + 0.01)
ax2.grid(alpha=0.3)
ax2.text(0.5, 0.05, f'Max drift: {np.max(np.abs(L_exact - L_exact[0])):.2e}',
         transform=ax2.transAxes, ha='center', fontsize=10, color='gray')

plt.suptitle('Conservation Laws — Numerical Reference Solution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_conservation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Numerical integrator conserves E to {np.max(np.abs(E_exact - E_exact[0])):.2e}')
print(f'Numerical integrator conserves L to {np.max(np.abs(L_exact - L_exact[0])):.2e}')

<a id='pinn-theory'></a>
## 🧠 6. PINN Architecture & Loss Function

The PINN takes scalar time $t$ as input and outputs 2D position $(\hat{x}, \hat{y})$:

$$\mathcal{N}: t \;\mapsto\; (\hat{x}(t),\; \hat{y}(t))$$

### Loss Function

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \lambda \mathcal{L}_{\text{physics}}$$

**Data loss** (fits to observed positions):
$$\mathcal{L}_{\text{data}} = \frac{1}{N}\sum_{i=1}^N \left[(\hat{x}(t_i)-x_i)^2 + (\hat{y}(t_i)-y_i)^2\right]$$

**Physics loss** (enforces Newton's gravity at $M$ collocation points):
$$\mathcal{L}_{\text{physics}} = \frac{1}{M}\sum_{j=1}^M \left[\left(\ddot{\hat{x}} + \frac{GM\hat{x}}{r^3}\right)^2 + \left(\ddot{\hat{y}} + \frac{GM\hat{y}}{r^3}\right)^2\right]$$

where $r = \sqrt{\hat{x}^2 + \hat{y}^2}$ and $\ddot{\hat{x}}, \ddot{\hat{y}}$ are computed via **automatic differentiation**.

<div style="background: #d1ecf1; border-left: 5px solid #17a2b8; padding: 15px; border-radius: 5px;">
<strong>💡 Why this is powerful:</strong> The collocation points span the <em>full</em> orbit — including regions with zero training data. By enforcing Newton's gravity everywhere, the PINN learns that the trajectory must be a closed ellipse, even without seeing the second half.
</div>

<a id='network'></a>
## 🏗️ 7. Network Implementation

In [ ]:
# ── Prepare tensors ────────────────────────────────────────────────────────────
t_tensor = torch.tensor(t_dense, dtype=torch.float32).view(-1, 1)
xy_full  = torch.tensor(np.stack([x_exact, y_exact], axis=1), dtype=torch.float32)

# Sparse training data (sample every 15th point from first 40% of orbit)
train_mask = t_dense <= END_TRAIN_FRAC * T_SIM
train_idx  = np.where(train_mask)[0][::15]
t_data     = t_tensor[train_idx]
xy_data    = xy_full[train_idx]

print(f'Total grid points    : {N_DENSE}')
print(f'Training data points : {len(train_idx)}  (first {END_TRAIN_FRAC*100:.0f}% of orbit, sparse)')

In [ ]:
class FCN(nn.Module):
    """
    Fully-connected PINN backbone.
    Input:  t  (scalar time)
    Output: (x, y) predicted position
    """
    def __init__(self, n_hidden: int = 64, n_layers: int = 4):
        super().__init__()
        layers = [nn.Linear(1, n_hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_hidden, n_hidden), nn.Tanh()]
        layers.append(nn.Linear(n_hidden, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


demo = FCN()
print(demo)
print(f'\nTotal trainable parameters: {demo.count_params():,}')

<a id='ann'></a>
## 📉 8. Standard NN Baseline

Trained on **data loss only** — same architecture, same training data, no physics.

In [ ]:
torch.manual_seed(42)
nn_model  = FCN(n_hidden=64, n_layers=4)
nn_optim  = torch.optim.Adam(nn_model.parameters(), lr=1e-3)
NN_EPOCHS = 5000
nn_losses = []

for epoch in range(NN_EPOCHS):
    nn_optim.zero_grad()
    pred = nn_model(t_data)
    loss = torch.mean((pred - xy_data)**2)
    loss.backward()
    nn_optim.step()
    nn_losses.append(loss.item())
    if (epoch + 1) % 1000 == 0:
        print(f'  Epoch {epoch+1:5d}  loss = {loss.item():.3e}')

print(f'\n✅ Standard NN trained  —  final loss: {nn_losses[-1]:.3e}')

In [ ]:
with torch.no_grad():
    xy_nn = nn_model(t_tensor).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x_exact, y_exact,      color=COLORS['exact'], lw=2.5, label='Exact orbit')
ax.plot(xy_nn[:, 0], xy_nn[:, 1], color=COLORS['nn'],   lw=2.5, ls='--', alpha=0.85,
        label='Standard NN')
ax.scatter(xy_data.numpy()[:, 0], xy_data.numpy()[:, 1], s=70, color=COLORS['data'],
           zorder=5, edgecolors='k', lw=0.5, label='Training data')
ax.scatter([0], [0], s=300, color='gold', marker='*', zorder=6, edgecolors='orange')
ax.set_aspect('equal')
ax.set_title('Orbital Trajectory — Standard NN', fontsize=12, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=10)

ax2 = axes[1]
ax2.plot(t_dense, x_exact,    color=COLORS['exact'], lw=2.5, label='x exact')
ax2.plot(t_dense, xy_nn[:,0], color=COLORS['nn'],   lw=2.5, ls='--', alpha=0.85, label='x NN')
ax2.axvspan(0, END_TRAIN_FRAC*T_SIM, alpha=0.08, color='orange', label='Training region')
ax2.set_title('x(t) — Standard NN fails to extrapolate', fontsize=12, fontweight='bold')
ax2.set_xlabel('Time'); ax2.set_ylabel('x')
ax2.legend(fontsize=10); ax2.grid(alpha=0.3)
ax2.annotate('Diverges here', xy=(END_TRAIN_FRAC*T_SIM + 0.5, xy_nn[int(0.55*N_DENSE), 0]),
             xytext=(END_TRAIN_FRAC*T_SIM + 0.3, -1.2), fontsize=10, color='red',
             arrowprops=dict(arrowstyle='->', color='red'))

plt.suptitle('Standard NN: Fails Beyond Training Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/03_nn_result.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='pinn'></a>
## 🚀 9. PINN Training

Now we add the Newton gravity ODE residual to the loss. Collocation points span the **full time domain** — so the physics constraint is active everywhere, including the unseen part of the orbit.

In [ ]:
N_COLLOC      = 50
LAMBDA_PHYS   = 1e-3
PINN_EPOCHS   = 20000
PINN_LR       = 5e-4

t_physics = torch.linspace(0, T_SIM, N_COLLOC).view(-1, 1).requires_grad_(True)

torch.manual_seed(42)
pinn_model       = FCN(n_hidden=64, n_layers=4)
pinn_optim       = torch.optim.Adam(pinn_model.parameters(), lr=PINN_LR)
pinn_losses      = []
pinn_data_losses = []
pinn_phys_losses = []

for epoch in range(PINN_EPOCHS):
    pinn_optim.zero_grad()

    # ── Data loss ─────────────────────────────────────────────────────────────
    pred_data = pinn_model(t_data)
    loss_data = torch.mean((pred_data - xy_data)**2)

    # ── Physics loss ──────────────────────────────────────────────────────────
    pred_phys = pinn_model(t_physics)           # (M, 2)
    xp = pred_phys[:, 0:1]                      # x̂(t)
    yp = pred_phys[:, 1:2]                      # ŷ(t)

    # First derivatives
    vx_p = torch.autograd.grad(xp, t_physics, torch.ones_like(xp), create_graph=True)[0]
    vy_p = torch.autograd.grad(yp, t_physics, torch.ones_like(yp), create_graph=True)[0]

    # Second derivatives (accelerations)
    ax_p = torch.autograd.grad(vx_p, t_physics, torch.ones_like(vx_p), create_graph=True)[0]
    ay_p = torch.autograd.grad(vy_p, t_physics, torch.ones_like(vy_p), create_graph=True)[0]

    # Gravitational accelerations from predicted position
    r_p      = torch.sqrt(xp**2 + yp**2 + 1e-8)
    ax_grav  = -GM * xp / r_p**3
    ay_grav  = -GM * yp / r_p**3

    # ODE residuals  (should be 0)
    res_x = ax_p - ax_grav
    res_y = ay_p - ay_grav
    loss_physics = torch.mean(res_x**2 + res_y**2)

    # ── Combined loss ─────────────────────────────────────────────────────────
    loss = loss_data + LAMBDA_PHYS * loss_physics
    loss.backward()
    pinn_optim.step()

    pinn_losses.append(loss.item())
    pinn_data_losses.append(loss_data.item())
    pinn_phys_losses.append(loss_physics.item())

    if (epoch + 1) % 5000 == 0:
        print(f'  Epoch {epoch+1:6d}  total={loss.item():.3e}  '
              f'data={loss_data.item():.3e}  phys={loss_physics.item():.3e}')

print(f'\n✅ PINN trained  —  final total loss: {pinn_losses[-1]:.3e}')

In [ ]:
with torch.no_grad():
    xy_pinn = pinn_model(t_tensor).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x_exact, y_exact,           color=COLORS['exact'], lw=2.5, label='Exact orbit')
ax.plot(xy_pinn[:,0], xy_pinn[:,1], color=COLORS['pinn'],  lw=2.5, ls='--', alpha=0.9, label='PINN')
ax.scatter(xy_data.numpy()[:,0], xy_data.numpy()[:,1], s=70, color=COLORS['data'],
           zorder=5, edgecolors='k', lw=0.5, label='Training data')
# Collocation points
tp_np = t_physics.detach().numpy().flatten()
with torch.no_grad():
    cp = pinn_model(t_physics.detach()).numpy()
ax.scatter(cp[:,0], cp[:,1], s=20, color=COLORS['physics'], alpha=0.5,
           label='Collocation pts', zorder=4)
ax.scatter([0],[0], s=300, color='gold', marker='*', zorder=6, edgecolors='orange')
ax.set_aspect('equal')
ax.set_title('PINN — Reconstructed Orbit', fontsize=12, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.plot(t_dense, x_exact,     color=COLORS['exact'], lw=2.5, label='x exact')
ax2.plot(t_dense, xy_pinn[:,0],color=COLORS['pinn'],  lw=2.5, ls='--', alpha=0.9, label='x PINN')
ax2.axvspan(0, END_TRAIN_FRAC*T_SIM, alpha=0.08, color='orange', label='Training region')
ax2.set_title('x(t) — PINN extrapolates correctly', fontsize=12, fontweight='bold')
ax2.set_xlabel('Time'); ax2.set_ylabel('x')
ax2.legend(fontsize=10); ax2.grid(alpha=0.3)
ax2.annotate('Physics guides\nextrapolation!', xy=(END_TRAIN_FRAC*T_SIM + 1.0, x_exact[int(0.65*N_DENSE)]),
             xytext=(END_TRAIN_FRAC*T_SIM + 0.6, -1.2), fontsize=10, color='green',
             arrowprops=dict(arrowstyle='->', color='green'))

plt.suptitle('PINN: Physics-Guided Orbit Reconstruction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/04_pinn_result.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='loss'></a>
## 📈 10. Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(nn_losses, color=COLORS['nn'], lw=1.5, label='NN data loss')
ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel('Loss (log scale)', fontsize=12)
ax.set_title('Standard NN — Convergence', fontsize=12, fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.semilogy(pinn_data_losses,  color=COLORS['data'],    lw=1.5, label='PINN data loss')
ax2.semilogy([LAMBDA_PHYS * v for v in pinn_phys_losses],
             color=COLORS['physics'], lw=1.5, label='λ × physics loss')
ax2.semilogy(pinn_losses,       color=COLORS['pinn'],    lw=2.5, ls='--', label='Total loss')
ax2.set_xlabel('Epoch', fontsize=12); ax2.set_ylabel('Loss (log scale)', fontsize=12)
ax2.set_title('PINN — Convergence', fontsize=12, fontweight='bold')
ax2.legend(fontsize=11); ax2.grid(alpha=0.3)

plt.suptitle('Loss Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/05_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

<a id='comparison'></a>
## 🔍 11. Final Comparison & Error Analysis

In [ ]:
err_nn   = np.sqrt((xy_nn[:,0]   - x_exact)**2 + (xy_nn[:,1]   - y_exact)**2)
err_pinn = np.sqrt((xy_pinn[:,0] - x_exact)**2 + (xy_pinn[:,1] - y_exact)**2)
split    = int(END_TRAIN_FRAC * N_DENSE)

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

# ─── A: trajectories ──────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(x_exact, y_exact,          color=COLORS['exact'], lw=2.5, label='Exact')
ax1.plot(xy_nn[:,0],   xy_nn[:,1],   color=COLORS['nn'],   lw=2.5, ls='-.', label='Standard NN')
ax1.plot(xy_pinn[:,0], xy_pinn[:,1], color=COLORS['pinn'], lw=2.5, ls='--', label='PINN')
ax1.scatter(xy_data.numpy()[:,0], xy_data.numpy()[:,1], s=70, color=COLORS['data'],
            zorder=5, edgecolors='k', lw=0.5, label='Training data')
ax1.scatter([0],[0], s=300, color='gold', marker='*', zorder=6, edgecolors='orange')
ax1.set_aspect('equal')
ax1.set_title('(A) Orbital Trajectories: Exact vs NN vs PINN', fontsize=13, fontweight='bold')
ax1.set_xlabel('x', fontsize=12); ax1.set_ylabel('y', fontsize=12)
ax1.legend(fontsize=11, framealpha=0.9)

# ─── B: position error over time ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
ax2.semilogy(t_dense, err_nn   + 1e-10, color=COLORS['nn'],   lw=2, label='NN |Δr|')
ax2.semilogy(t_dense, err_pinn + 1e-10, color=COLORS['pinn'], lw=2, label='PINN |Δr|')
ax2.axvline(x=t_dense[split], color='gray', ls=':', lw=1.5, label='End of training data')
ax2.set_title('(B) Position Error Over Time', fontsize=12, fontweight='bold')
ax2.set_xlabel('Time', fontsize=11); ax2.set_ylabel('|Δr| (log)', fontsize=11)
ax2.legend(fontsize=10); ax2.grid(alpha=0.3)

# ─── C: bar chart ─────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
labels    = ['Training region\n(0–40% orbit)', 'Extrapolation\n(40–100% orbit)']
nn_bars   = [err_nn[:split].mean(),   err_nn[split:].mean()]
pinn_bars = [err_pinn[:split].mean(), err_pinn[split:].mean()]
x_bars    = np.arange(2)
w = 0.35
b1 = ax3.bar(x_bars - w/2, nn_bars,   width=w, color=COLORS['nn'],   label='NN',   alpha=0.85)
b2 = ax3.bar(x_bars + w/2, pinn_bars, width=w, color=COLORS['pinn'], label='PINN', alpha=0.85)
ax3.set_xticks(x_bars); ax3.set_xticklabels(labels, fontsize=10)
ax3.set_ylabel('Mean |Δr|', fontsize=11)
ax3.set_title('(C) Mean Position Error by Region', fontsize=12, fontweight='bold')
ax3.legend(fontsize=11)
for bar in ax3.patches:
    if bar.get_height() > 0:
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Kepler Orbit — NN vs PINN Full Comparison', fontsize=14, fontweight='bold')
plt.savefig('plots/06_full_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n── Error Summary ────────────────────────────────────────────────')
print(f'{"":18s}  {"Train MAE":>10}  {"Extrapol. MAE":>14}')
print(f'{"Standard NN":18s}  {nn_bars[0]:>10.5f}  {nn_bars[1]:>14.5f}')
print(f'{"PINN":18s}  {pinn_bars[0]:>10.5f}  {pinn_bars[1]:>14.5f}')
print(f'\nPINN improvement (extrapolation): {nn_bars[1]/max(pinn_bars[1],1e-9):.1f}×')

<a id='conservation-check'></a>
## 💎 12. Conservation Law Verification

A key test for a physics-informed model: do the predictions **conserve energy and angular momentum**? The PINN was never explicitly told to conserve these — but they follow from the ODE it was constrained to satisfy.

In [ ]:
# ── Compute velocities of PINN via numerical differentiation ──────────────────
dt = t_dense[1] - t_dense[0]
vx_pinn = np.gradient(xy_pinn[:, 0], dt)
vy_pinn = np.gradient(xy_pinn[:, 1], dt)
r_pinn  = np.sqrt(xy_pinn[:,0]**2 + xy_pinn[:,1]**2 + 1e-12)
E_pinn  = 0.5*(vx_pinn**2 + vy_pinn**2) - GM/r_pinn
L_pinn  = xy_pinn[:,0]*vy_pinn - xy_pinn[:,1]*vx_pinn

vx_nn   = np.gradient(xy_nn[:,0], dt)
vy_nn   = np.gradient(xy_nn[:,1], dt)
r_nn    = np.sqrt(xy_nn[:,0]**2 + xy_nn[:,1]**2 + 1e-12)
E_nn    = 0.5*(vx_nn**2 + vy_nn**2) - GM/r_nn
L_nn    = xy_nn[:,0]*vy_nn - xy_nn[:,1]*vx_nn

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for row, (label, E_m, L_m, color) in enumerate([
    ('Standard NN', E_nn,   L_nn,   COLORS['nn']),
    ('PINN',        E_pinn, L_pinn, COLORS['pinn'])
]):
    ax = axes[row, 0]
    ax.plot(t_dense, E_exact, color=COLORS['exact'], lw=2, label='Exact E', alpha=0.7)
    ax.plot(t_dense, E_m,     color=color,           lw=2, label=f'{label} E', ls='--')
    ax.axvline(t_dense[split], color='gray', ls=':', lw=1)
    ax.set_title(f'Energy — {label}', fontweight='bold', fontsize=11)
    ax.set_xlabel('Time'); ax.set_ylabel('Total Energy E')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax2 = axes[row, 1]
    ax2.plot(t_dense, L_exact, color=COLORS['exact'], lw=2, label='Exact L', alpha=0.7)
    ax2.plot(t_dense, L_m,     color=color,           lw=2, label=f'{label} L', ls='--')
    ax2.axvline(t_dense[split], color='gray', ls=':', lw=1)
    ax2.set_title(f'Angular Momentum — {label}', fontweight='bold', fontsize=11)
    ax2.set_xlabel('Time'); ax2.set_ylabel('Ang. Momentum L')
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.suptitle('Conservation Laws: Does the Model Respect Physics?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/07_conservation_check.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nEnergy conservation (extrapolation region):')
print(f'  NN   std(E) : {E_nn[split:].std():.4f}')
print(f'  PINN std(E) : {E_pinn[split:].std():.4f}')
print('\nAngular momentum conservation (extrapolation region):')
print(f'  NN   std(L) : {L_nn[split:].std():.4f}')
print(f'  PINN std(L) : {L_pinn[split:].std():.4f}')

<a id='animation'></a>
## 🎬 13. Training Animation

In [ ]:
SNAP_EVERY  = 1000
ANIM_EPOCHS = 10000

torch.manual_seed(42)
anim_model  = FCN(n_hidden=64, n_layers=4)
anim_optim  = torch.optim.Adam(anim_model.parameters(), lr=5e-4)
snapshots   = []

for epoch in range(ANIM_EPOCHS):
    anim_optim.zero_grad()
    pred_data   = anim_model(t_data)
    loss_data   = torch.mean((pred_data - xy_data)**2)
    pred_phys   = anim_model(t_physics)
    xp = pred_phys[:,0:1]; yp = pred_phys[:,1:2]
    vx_p = torch.autograd.grad(xp, t_physics, torch.ones_like(xp), create_graph=True)[0]
    vy_p = torch.autograd.grad(yp, t_physics, torch.ones_like(yp), create_graph=True)[0]
    ax_p = torch.autograd.grad(vx_p, t_physics, torch.ones_like(vx_p), create_graph=True)[0]
    ay_p = torch.autograd.grad(vy_p, t_physics, torch.ones_like(vy_p), create_graph=True)[0]
    r_p  = torch.sqrt(xp**2 + yp**2 + 1e-8)
    res_x = ax_p + GM*xp/r_p**3; res_y = ay_p + GM*yp/r_p**3
    loss_phys = torch.mean(res_x**2 + res_y**2)
    loss = loss_data + LAMBDA_PHYS * loss_phys
    loss.backward(); anim_optim.step()

    if (epoch + 1) % SNAP_EVERY == 0:
        with torch.no_grad():
            snap = anim_model(t_tensor).numpy()
        snapshots.append((epoch + 1, snap.copy()))

# ── Build animation ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1, ax2 = axes
ax1.plot(x_exact, y_exact, color=COLORS['exact'], lw=2, label='Exact', alpha=0.8)
ax1.scatter(xy_data.numpy()[:,0], xy_data.numpy()[:,1], s=60, color=COLORS['data'],
            zorder=5, edgecolors='k', lw=0.5)
ax1.scatter([0],[0], s=200, color='gold', marker='*', zorder=6, edgecolors='orange')
pinn_line1, = ax1.plot([], [], color=COLORS['pinn'], lw=2.5, ls='--', label='PINN')
ax1.set_aspect('equal'); ax1.set_xlabel('x'); ax1.set_ylabel('y')
ax1.legend(fontsize=10)

ax2.plot(t_dense, x_exact, color=COLORS['exact'], lw=2, alpha=0.8)
pinn_line2, = ax2.plot([], [], color=COLORS['pinn'], lw=2.5, ls='--')
ax2.axvspan(0, END_TRAIN_FRAC*T_SIM, alpha=0.08, color='orange')
ax2.set_xlabel('Time'); ax2.set_ylabel('x')

title = fig.suptitle('', fontsize=13, fontweight='bold')

def update(frame):
    epoch, snap = snapshots[frame]
    pinn_line1.set_data(snap[:,0], snap[:,1])
    pinn_line2.set_data(t_dense,   snap[:,0])
    title.set_text(f'PINN Training — Epoch {epoch:,}')
    return pinn_line1, pinn_line2, title

anim = FuncAnimation(fig, update, frames=len(snapshots), interval=300, blit=False)
plt.tight_layout()
plt.close()

HTML(anim.to_jshtml())

<a id='conclusions'></a>
## 🏁 14. Conclusions

<div style="background: linear-gradient(135deg, #0a0a1a 0%, #1b2838 100%); color: white; padding: 25px; border-radius: 12px;">

### Summary

| | Standard NN | PINN |
|---|---|---|
| **Physics knowledge** | None | Newton's gravity |
| **Training data** | Sparse (40% orbit) | Sparse (40% orbit) |
| **Orbit reconstruction** | ❌ Fails to close | ✅ Correct ellipse |
| **Energy conservation** | ❌ Violated | ✅ Approximately preserved |
| **Angular momentum** | ❌ Violated | ✅ Approximately preserved |

### Key Insights

1. **The physics residual acts as global regularisation**: even with data from only 40% of the orbit, the PINN correctly reconstructs the full ellipse.

2. **Conservation laws emerge for free**: we never explicitly enforced $E = \text{const}$ or $L = \text{const}$ — yet the PINN approximately satisfies them because they are consequences of the ODE it was trained on.

3. **Collocation points are key**: by placing them across the full time domain, we give the physics constraint maximum leverage.

### Extensions
- Add velocity observations to the training data
- Try perturbed orbits (3-body problem)
- Enforce initial/boundary conditions exactly via **hard constraints**
- Use adaptive physics weight $\lambda$ (curriculum learning)

### References
- Raissi, Perdikaris, Karniadakis (2019). *Physics-informed neural networks.* J. Comput. Phys.
- Cranmer et al. (2020). *Lagrangian Neural Networks.*
- Greydanus et al. (2019). *Hamiltonian Neural Networks.*

</div>